# 使用 Bedrock AgentCore Gateway 从 OpenAPI 规范构建 MCP

本动手实验演示如何使用 Amazon Bedrock AgentCore Gateway 从 OpenAPI 规范自动生成 Model Context Protocol (MCP) 服务器，实现外部 API 与 AI 代理的无缝集成。

## 概述

在本实验中，您将：
- 创建带有身份验证的 Bedrock AgentCore Gateway
- 使用外部 API 规范配置基于 OpenAPI 的网关目标
- 设置安全的凭证管理以访问外部 API
- 部署并使用 Strands Agents 测试网关
- 探索从 OpenAPI 模式自动生成 MCP 工具

## 前提条件

在开始本实验之前，请确保您已具备：
- 已配置 AWS 凭证（IAM 角色或环境变量）
- 已安装所需的 Python 包
- 基于 AWS 区域的 Nova Pro 模型 ID
- 用于测试的外部 API 密钥（例如 Exa API 密钥）

如果您未在已承担 IAM 角色的环境中运行，请将 AWS 凭证设置为环境变量：

In [ ]:
import os

#os.environ["AWS_ACCESS_KEY_ID"]=<YOUR ACCESS KEY>
#os.environ["AWS_SECRET_ACCESS_KEY"]=<YOUR SECRET KEY>
#os.environ["AWS_SESSION_TOKEN"]=<OPTIONAL - YOUR SESSION TOKEN IF TEMP CREDENTIAL>
#os.environ["AWS_REGION"]=<AWS REGION WITH BEDROCK AGENTCORE AVAILABLE>

安装 Strands Agents 和 Bedrock AgentCore Python SDK 所需的包：

In [ ]:
#%pip install -q strands-agents strands-agents-tools bedrock-agentcore rich

根据 AWS 区域设置 Nova Pro 模型 ID：

In [ ]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

## 什么是 Bedrock AgentCore Gateway？

Amazon Bedrock AgentCore Gateway 是一项托管服务，可自动将 OpenAPI 规范转换为 Model Context Protocol (MCP) 服务器。主要优势包括：

- **自动工具生成**：无需手动编码即可将 OpenAPI 端点转换为 MCP 工具
- **安全身份验证**：内置支持多种身份验证方法（API 密钥、JWT、OAuth）
- **托管基础设施**：无需部署或维护自定义 MCP 服务器
- **可扩展性**：根据需求自动扩展
- **集成能力**：与现有 REST API 无缝集成

这种方法使您能够快速将任何具有 OpenAPI 文档的 REST API 作为工具集成到您的 AI 代理中。

![openapi-gateway-apikey.png](images/openapi-gateway-apikey.png)

## 创建以 OpenAPI 为目标的 AgentCore Gateway

### 步骤 1：准备 OpenAPI 规范

下载 Exa API 的 OpenAPI 规范，该规范将用于自动生成 MCP 工具。OpenAPI 规范定义了所有可用的端点、参数和响应模式。

In [ ]:
import requests
import os

url = "https://raw.githubusercontent.com/exa-labs/openapi-spec/refs/heads/master/exa-openapi-spec.yaml"
openapi_file_name = url.split("/")[-1]
save_path = f"./{openapi_file_name}"

if os.path.exists(save_path):
    print(f"File already exists: {save_path}")
else:
    response = requests.get(url)

    if response.status_code == 200:
        with open(save_path, 'wb') as file:
            file.write(response.content)
        print(f"File downloaded successfully: {save_path}")
    else:
        print(f"Failed to download file. Status code: {response.status_code}")

### 步骤 2：将 OpenAPI 规范上传到 S3

将 OpenAPI 规范上传到 S3，以便 AgentCore Gateway 可以访问它来自动生成工具。

In [ ]:
import boto3

region = boto3.session.Session().region_name
# Create an S3 client
s3_client = boto3.client('s3', region_name=region)
sts_client = boto3.client('sts', region_name=region)

account_id = sts_client.get_caller_identity()["Account"]

# Define parameters
# Your s3 bucket to upload the OpenAPI json file.
bucket_name = f'bedrock-agentcore-gateway-{account_id}-{region}'
file_path = f'./{openapi_file_name}'
object_key = openapi_file_name

# Upload the file using put_object and read response
try:
    if region == "us-east-1":
        s3bucket = s3_client.create_bucket(
            Bucket=bucket_name
        )
    else:
        s3bucket = s3_client.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={
                'LocationConstraint': region
            }
        )
    with open(file_path, 'rb') as file_data:
        response = s3_client.put_object(
            Bucket=bucket_name,
            Key=object_key,
            Body=file_data
        )

    # Construct the ARN of the uploaded object with account ID and region
    openapi_s3_uri = f's3://{bucket_name}/{object_key}'
    print(f'Uploaded object S3 URI: {openapi_s3_uri}')
except Exception as e:
    print(f'Error uploading file: {e}')
    with open(file_path, 'rb') as file_data:
        response = s3_client.put_object(
            Bucket=bucket_name,
            Key=object_key,
            Body=file_data
        )
    # Construct the ARN of the uploaded object with account ID and region
    openapi_s3_uri = f's3://{bucket_name}/{object_key}'
    print(f'Uploaded object S3 URI: {openapi_s3_uri}')

### 步骤 3：创建带有入站身份验证的 AgentCore Gateway

首先，创建一个 Cognito 用户池以安全访问 AgentCore Gateway。这为网关端点提供基于 JWT 的身份验证。

**创建的组件：**
- **用户池**：管理用户身份和身份验证
- **应用客户端**：启用应用程序级别的身份验证
- **Cognito 托管域**：为 OAuth 2.0 令牌端点提供托管域

然后创建带有 Cognito JWT 身份验证的主 AgentCore Gateway。该网关将托管我们基于 OpenAPI 的目标。

In [ ]:
from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient
import boto3

region = boto3.session.Session().region_name

gateway_client = GatewayClient(region_name=region)

# Creating a cognito OAuth authorizer 
cognito_response = gateway_client.create_oauth_authorizer_with_cognito("agentcore-gateway")

cognito_pool_id = cognito_response['client_info']['user_pool_id']
cognito_client_id = cognito_response['client_info']['client_id']
cognito_client_secret = cognito_response['client_info']['client_secret']
cognito_scope = cognito_response['client_info']['scope']
cognito_token_endpoint = cognito_response['client_info']['token_endpoint']
cognito_domain = cognito_response['client_info']['domain_prefix']
print(f"✅ User Pool ID: {cognito_pool_id}")
print(f"✅ Client ID: {cognito_client_id}")
print(f"✅ Client Secret: {cognito_client_secret}")
print(f"✅ Scope: {cognito_scope}")
print(f"✅ Token Endpoint: {cognito_token_endpoint}")
print(f"✅ Cognito Domain: {cognito_domain}")

# Creating a gatweay using the cognito authorizer for its access
gateway = gateway_client.create_mcp_gateway(authorizer_config=cognito_response["authorizer_config"])

gateway_id = gateway['gatewayId']
gateway_url = gateway['gatewayUrl']
print(f"✅ Gateway ID: {gateway_id}")
print(f"✅ Gateway URL: {gateway_url}")

### 步骤 4：创建带有 OpenAPI 配置和出站身份验证的网关目标

创建一个使用 OpenAPI 规范自动生成 MCP 工具的网关目标，同时 API 密钥安全地存储在 Bedrock AgentCore Identity API Key Credential Provider 中用于出站身份验证。目标配置包括：

- **OpenAPI 模式**：引用存储在 S3 中的 OpenAPI 规范
- **凭证配置**：如何对外部 API 进行身份验证
- **参数映射**：API 密钥的放置位置（查询参数、请求头等）


要获取 Exa API 密钥，请前往 [Exa 登录页面](https://dashboard.exa.ai/login) 使用您的电子邮件注册。

然后前往 Exa 仪表板中的 [API 密钥部分](https://dashboard.exa.ai/api-keys) 创建 API 密钥。将 API 密钥复制到下方代码中的 `EXA_API_KEY`...

⚠️ **重要提示**：请将占位符 API 密钥替换为您的实际 Exa API 密钥。

In [ ]:
# !-------- UPDATE THE EXA API KEY HERE  --------!
EXA_API_KEY = <YOUR EXA API KEY> 

gateway_target = gateway_client.create_mcp_gateway_target(
    gateway=gateway, 
    target_type="openApiSchema", 
    target_payload={
        "s3": {
            "uri": openapi_s3_uri
        }
    },
    credentials={
        # !-------- UPDATE THE EXA API KEY HERE  --------!
        "api_key": EXA_API_KEY, 
        "credential_location": "HEADER",
        "credential_parameter_name": "x-api-key"
    }
)
gateway_target_id = gateway_target['targetId']
credential_provider_name = gateway_target['credentialProviderConfigurations'][0]['credentialProvider']['apiKeyCredentialProvider']['providerArn'].split('/')[-1]

print(f"✅ Gateway Target ID: {gateway_id}")
print(f"✅ API Key Credential Provider Name: {credential_provider_name}")

## 使用 Strands Agent 测试已部署的 AgentCore Gateway MCP 服务器

现在让我们使用正确的身份验证来测试已部署的 AgentCore Gateway MCP 服务器。


### 从 Cognito 身份验证获取访问令牌

从 Cognito 获取 JWT 访问令牌，该令牌将用于对 AgentCore Gateway 的请求进行身份验证。

In [ ]:
# Get access token from Cognito
client_config = {
    "user_pool_id": cognito_pool_id,
    "client_id": cognito_client_id,
    "client_secret": cognito_client_secret,
    "scope": cognito_scope,
    "token_endpoint": cognito_token_endpoint,
    "region": region
}

token_response = gateway_client.get_access_token_for_cognito(client_config)
access_token = token_response
print(access_token)

### 使用 Strands Agent 测试网关

现在让我们通过将 Strands Agent 连接到 AgentCore Gateway 来测试它。网关会自动将 OpenAPI 规范转换为代理可以使用的 MCP 工具。

**此处发生的操作：**
1. 使用 JWT Bearer 令牌连接到网关
2. 列出可用工具（从 OpenAPI 规范自动生成）
3. 创建一个可以访问这些工具的代理
4. 测试代理通过网关使用外部 API 的能力

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

# Connect to the Web Search MCP server
print("\n正在连接到 MCP 服务器...")
headers = {
    "Authorization": f"Bearer {access_token}",
    #"Content-Type": "application/json"
}
exa_server = MCPClient(lambda: streamablehttp_client(gateway_url, headers))

with exa_server:
    mcp_tools = (exa_server.list_tools_sync())
    print(f"Available tools: {[tool.tool_name for tool in mcp_tools]}")

    # Create agent with self-built MCP tools
    agent = Agent(
        model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
        system_prompt = """你是一个生活助手，运用网络搜索的知识回答各种问题。""",
        tools=mcp_tools,
    )

    agent("什么是 Amazon Bedrock AgentCore？")

让我们查看代理循环的详细执行流程，以了解代理如何处理请求并生成响应：

In [ ]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta")
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan")
table.add_column("Tool Result", style="cyan")

for message in agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(message["role"], text[-1] if text else "", 
                  tool_name[-1] if tool_name else "", 
                  json.dumps(tool_input[-1], indent=2) if tool_input else "", 
                  (json.dumps(tool_result[-1], indent=2)[:500]+"\n.\n.\n." if len(str(tool_result[-1])) > 500 else json.dumps(tool_result[-1], indent=2)) if tool_result else "")

console.print(table)

## 资源清理（可选）

清理已部署的资源：

In [ ]:
import boto3
import os

region = boto3.session.Session().region_name

agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)
cognito_client = boto3.client('cognito-idp', region_name=region)
iam_client = boto3.client('iam')
s3_client = boto3.client('s3', region_name=region)

try:
    print("Deleting AgentCore Gateway Target...")
    agentcore_control_client.delete_gateway_target(gatewayIdentifier=gateway_id, targetId=gateway_target_id)
    print("✓ AgentCore Gateway Target deleted")
    
    print("Deleting AgentCore Gateway...")
    agentcore_control_client.delete_gateway(gatewayIdentifier=gateway_id)
    print("✓ AgentCore Gateway deletion initiated")

    print("Deleting AgentCore Identity...")
    agentcore_control_client.delete_api_key_credential_provider(name=credential_provider_name)
    print("✓ AgentCore Identity deletion initiated")

    print("Deleting Cognito User Pool...")
    cognito_client.delete_user_pool_domain(Domain=cognito_response['client_info']['domain_prefix'], UserPoolId=cognito_pool_id)
    cognito_client.delete_user_pool(UserPoolId=cognito_pool_id)
    print("✓ Cognito User Pool deleted")

    print("Deleting S3 Bucket...")
    s3_client.delete_object(Bucket=bucket_name, Key=openapi_file_name)
    s3_client.delete_bucket(Bucket=bucket_name)
    print("✓ S3 Bucket deleted")
except Exception as e:
    print(f"❌ Error during cleanup: {e}")
    print("You may need to manually clean up some resources.")

## 总结

在本实验中，您成功完成了：

- ✅ 创建了带有 JWT 身份验证的 Bedrock AgentCore Gateway
- ✅ 配置了外部 API 的安全凭证管理
- ✅ 从 OpenAPI 规范自动生成了 MCP 工具
- ✅ 将网关与 Strands Agents 集成，实现 AI 驱动的 API 交互

## AgentCore Gateway 的主要优势

- **零代码 MCP 生成**：自动将任何 OpenAPI 规范转换为 MCP 工具
- **安全凭证管理**：内置支持多种身份验证方法
- **托管基础设施**：无需部署或维护自定义服务器
- **可扩展**：自动处理负载和扩展
- **基于标准**：适用于任何具有 OpenAPI 文档的 REST API